In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RealTimeTaxi")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    .getOrCreate()
)

ImportError: cannot import name 'StreamingQueryException' from 'pyspark.errors' (unknown location)

In [2]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "latest")
    .load()
    )

In [3]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    DoubleType
)

taxi_schema = StructType([
    StructField("event_id", StringType(), True),

    StructField("VendorID", IntegerType(), True),

    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),

    StructField("RatecodeID", DoubleType(), True),

    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),

    StructField("payment_type", IntegerType(), True),

    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),

    StructField("total_time_in_sec", DoubleType(), True),

    StructField("ptime", StringType(), True),
    StructField("dtime", StringType(), True)
])

In [4]:
from pyspark.sql.functions import col, from_json

taxi_stream = (
    raw_stream
    .select(
        col("value").cast("string").alias("json")
    )
    .select(
        from_json(col("json"), taxi_schema).alias("data")
    )
    .select("data.*")
)

In [5]:
from pyspark.sql.functions import to_timestamp

taxi_stream = (
    taxi_stream
    .withColumn("ptime", to_timestamp("ptime"))
    .withColumn("dtime", to_timestamp("dtime"))
)
taxi_stream.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- total_time_in_sec: double (nullable = true)
 |-- ptime: timestamp (nullable = true)
 |-- dtime: timestamp (nullable = true)



In [6]:
for q in spark.streams.active:
    q.stop()

In [7]:
import os
import shutil

checkpoint = r"D:\projects\Real_time_Taxi\checkpoints\test_console"

if os.path.exists(checkpoint):
    shutil.rmtree(checkpoint)

In [8]:
query = (
    taxi_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 10)
    .option("checkpointLocation", checkpoint)
    .start()
)

Py4JJavaError: An error occurred while calling o77.start.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.DelegateToFileSystem.listStatus(DelegateToFileSystem.java:182)
	at org.apache.hadoop.fs.ChecksumFs.listStatus(ChecksumFs.java:571)
	at org.apache.hadoop.fs.FileContext$Util$1.next(FileContext.java:1934)
	at org.apache.hadoop.fs.FileContext$Util$1.next(FileContext.java:1930)
	at org.apache.hadoop.fs.FSLinkResolver.resolve(FSLinkResolver.java:90)
	at org.apache.hadoop.fs.FileContext$Util.listStatus(FileContext.java:1936)
	at 